# Bribri TTS: MMS-TTS Fine-Tuning

COSC 72 Final Project — Spring 2026

Ikenna Nwafor & Zachary Shah

Ikenna.27@dartmouth.edu

Zachary.s.shah.27@dartmouth.edu

Fine-tunes a pretrained VITS model (MMS-TTS) on ~25 minutes of Bribri audio.
The base model is configurable: switch between `facebook/mms-tts-cjp` (Cabécar,
closely related to Bribri) and `facebook/mms-tts-spa` (Spanish, much larger
pretraining set) via the `BASE_MODEL` variable below.

## Notebook Source
MMS-TTS does not support fine-tuning out of the box, so we employ the help of an existing project from Yoach Lacombe at https://github.com/ylacombe/finetune-hf-vits.git

## Patches
There are quite a few version complications and subsequent patches that must be run in order to get this to run! We have boiled it down to the minimum required to produce an output.

### 0. Setup

To resolve dependency issues that arise with using an older package, we must downgrade certain installs to the newest versions that still work cleanly. After running cell `0.1` you must RESTART THE RUNTIME to apply the changes.

In [ ]:
# 0.1 — Pinned installs. Run once, then RESTART RUNTIME before moving on.
# transformers 4.45.2 is the last version that works cleanly with finetune-hf-vits's
# trainer subclass; newer versions changed signatures. numpy 1.26 avoids the np.array
# copy=False breakage and the numba<2.1 requirement.
!pip install -q --force-reinstall "transformers==4.45.2" "datasets==2.20.0" "tokenizers<0.21" "accelerate==0.34.2"
!pip install -q "numpy==1.26.4" av
print("Done. Now: Runtime → Restart session, then continue from cell 0.2.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# 0.2 — Imports (run after restart)
import os, json, shutil, subprocess, unicodedata, glob, re
import torch
import torch.nn as nn
import pandas as pd
import soundfile as sf
import scipy.io.wavfile
from transformers import VitsModel, AutoTokenizer, VitsConfig
from safetensors.torch import load_file, save_file
from datasets import Dataset, DatasetDict, Audio, load_from_disk
import IPython.display as ipd

### 1. Configuration

To switch between cjp and spa experimentation, just comment out `BASE_MODEL` of the one you are not using.

In [ ]:
# 1.1 — Mount Drive
from google.colab import drive
drive.flush_and_unmount()
# !rm -rf /content/drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# 1.2 — Constants. Switch BASE_MODEL to swap experiments.
BASE_MODEL = 'facebook/mms-tts-cjp'   # Cabécar (Chibchan family, close to Bribri)
# BASE_MODEL = 'facebook/mms-tts-spa'     # Spanish (much larger pretraining set)

MODEL_TAG = BASE_MODEL.split('-')[-1]   # 'cjp' or 'spa', used in output paths

# Drive (persistent across Colab disconnects)
DRIVE_ROOT = '/content/drive/MyDrive/Junior/cosc72'
CSV_DIR    = f'{DRIVE_ROOT}/csv'
WAV_DIR    = f'{DRIVE_ROOT}/wav'
OUT_DIR    = f'{DRIVE_ROOT}/outputs/{MODEL_TAG}'

# Per-experiment paths (Drive)
BASE_CKPT_DIR    = f'{OUT_DIR}/base_model_extended'   # generator only, 49/51-vocab
TRAIN_BASE_DIR   = f'{OUT_DIR}/base_model_with_disc'  # generator + discriminator
DATASET_DIR      = f'{OUT_DIR}/dataset'               # HF arrow dataset
CONFIG_PATH      = f'{OUT_DIR}/training_config.json'
TRAIN_OUTPUT_DIR = f'{OUT_DIR}/training_run'
SAMPLES_DIR      = f'{OUT_DIR}/samples_for_eval'      # 20 wav files for human eval

# Local (Colab VM disk; cleared on disconnect)
REPO_DIR       = '/content/finetune-hf-vits'
TEMP_DISC_DIR  = '/content/temp_discriminator'

SEED = 42
torch.manual_seed(SEED)

os.makedirs(OUT_DIR, exist_ok=True)
print(f"BASE_MODEL  = {BASE_MODEL}")
print(f"MODEL_TAG   = {MODEL_TAG}")
print(f"OUT_DIR     = {OUT_DIR}")

### 2. Load and validate data

Reads the three CSVs, rewrites the `path` column from the original workshop
path to our `WAV_DIR`, verifies every audio file exists, and applies NFD
Unicode normalization to the transcripts so the tone diacritics become separate
combining-mark codepoints (which we'll tokenize as separate tokens).

In [ ]:
# 2.1 — Load splits, rewrite paths, NFD-normalize
def normalize_transcript(s):
    s = unicodedata.normalize('NFD', s)
    s = s.replace('0', '')   # drop stray digits used as transcription noise marks
    return s

splits = {}
for name in ['train', 'valid', 'test']:
    df = pd.read_csv(f'{CSV_DIR}/bribri-wav2vec2-{name}.csv')
    df['path']         = df['path'].apply(lambda p: os.path.join(WAV_DIR, os.path.basename(p)))
    df['exists']       = df['path'].apply(os.path.exists)
    df['sentence_nfd'] = df['sentence'].apply(normalize_transcript)
    missing = (~df['exists']).sum()
    print(f"{name}: {len(df)} rows, {missing} missing files")
    splits[name] = df[df['exists']].drop(columns=['exists']).reset_index(drop=True)

print(f"\nExample (raw → NFD):")
print(f"  raw: {splits['train']['sentence'].iloc[0]!r}")
print(f"  nfd: {splits['train']['sentence_nfd'].iloc[0]!r}")

### 3. Repo setup and patches

Now, we clone `ylacombe/finetune-hf-vits`, build the Monotonic Alignment Search (MAS)
Cython extension that VITS uses for text-audio alignment during training, and
apply all the patches the script needs to work with the current versions of
transformers / matplotlib / numpy:

- `plot.py`: `fig.canvas.tostring_rgb()` was removed in matplotlib 3.9; switch to `buffer_rgba()`.
- `utils/modeling_vits_training.py`: `config.pad_token_id` → `getattr(config, 'pad_token_id', None)` so the disc converter works for MMS configs that don't define pad_token_id.
- `run_vits_finetuning.py`:
  - import `load_from_disk` and wrap `load_dataset` calls in a fallback that reads our local saved dataset format
  - `parser.parse_json_file(..., allow_extra_keys=True)` so extra fields in our config don't error
  - `from transformers.utils import send_example_telemetry` wrapped in try/except (removed in newer transformers)
  - `getattr(training_args, "overwrite_output_dir", True)` instead of direct attribute access
  - `batch.get("speaker_id")` instead of `batch["speaker_id"]` (we have no speaker IDs)

In [ ]:
# 3.1 — Clone repo and build MAS
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/ylacombe/finetune-hf-vits.git', REPO_DIR], check=True)
    print("Cloned finetune-hf-vits")
else:
    print("Repo already cloned")

# Build MAS Cython extension. The setup script needs a monotonic_align/ subdir to exist
# inside monotonic_align/ before it can build.
mas_dir = f'{REPO_DIR}/monotonic_align'
os.makedirs(f'{mas_dir}/monotonic_align', exist_ok=True)
subprocess.run(['python', 'setup.py', 'build_ext', '--inplace'], cwd=mas_dir, capture_output=True, check=True)
so_files = glob.glob(f'{mas_dir}/**/*.so', recursive=True)
print(f"MAS built. .so files: {len(so_files)}")

In [ ]:
# 3.2 — Apply all patches to the cloned repo (idempotent)

# Patch a: utils/plot.py — matplotlib 3.9+ removed tostring_rgb
plot_path = f'{REPO_DIR}/utils/plot.py'
with open(plot_path) as f:
    pc = f.read()
pc_new, n_plot = re.subn(
    r'(\s+)data = np\.frombuffer\(fig\.canvas\.tostring_rgb\(\), dtype=np\.uint8\)',
    lambda m: f'{m.group(1)}fig.canvas.draw()\n{m.group(1)}data = np.asarray(fig.canvas.buffer_rgba())[:, :, :3].flatten().astype(np.uint8)',
    pc,
)
if n_plot:
    with open(plot_path, 'w') as f:
        f.write(pc_new)
print(f"  plot.py: {n_plot} patch(es)")

# Patch b: utils/modeling_vits_training.py — pad_token_id safe access for disc conversion
mvt_path = f'{REPO_DIR}/utils/modeling_vits_training.py'
with open(mvt_path) as f:
    mvt = f.read()
mvt_new = re.sub(r'(?<!getattr\()config\.pad_token_id(?!\s*,)', "getattr(config, 'pad_token_id', None)", mvt)
if mvt_new != mvt:
    with open(mvt_path, 'w') as f:
        f.write(mvt_new)
    print("  modeling_vits_training.py: patched pad_token_id")
else:
    print("  modeling_vits_training.py: already patched")

# Patch c: run_vits_finetuning.py — multiple
script_path = f'{REPO_DIR}/run_vits_finetuning.py'
with open(script_path) as f:
    s = f.read()
orig = s

# c.1: speaker_id → batch.get
s = s.replace('speaker_id=batch["speaker_id"],', 'speaker_id=batch.get("speaker_id"),')

# c.2: overwrite_output_dir attribute access → getattr
s = s.replace('not training_args.overwrite_output_dir', 'not getattr(training_args, "overwrite_output_dir", True)')

# c.3: parse_json_file → allow_extra_keys=True
s = re.sub(
    r'parser\.parse_json_file\(json_file=os\.path\.abspath\(sys\.argv\[1\]\)\)',
    'parser.parse_json_file(json_file=os.path.abspath(sys.argv[1]), allow_extra_keys=True)',
    s,
)

# c.4: send_example_telemetry stub for newer transformers
if 'try:\n    from transformers.utils import send_example_telemetry' not in s:
    s = s.replace(
        'from transformers.utils import send_example_telemetry',
        'try:\n    from transformers.utils import send_example_telemetry\nexcept ImportError:\n    def send_example_telemetry(*args, **kwargs): pass',
    )

# c.5: load_from_disk import + fallback wrap around load_dataset assignments
if 'load_from_disk' not in s:
    s = s.replace('from datasets import load_dataset', 'from datasets import load_dataset, load_from_disk')

# Wrap every "X = load_dataset(...)" assignment with a conditional that checks for
# our local saved-dataset format first. Uses paren-matching to find the close.
new_parts, last_end = [], 0
for m in re.finditer(r'([\w\[\]"\']+)\s*=\s*load_dataset\(', s):
    lhs, start = m.group(1), m.end()
    depth, i = 1, start
    while i < len(s) and depth > 0:
        if s[i] == '(':
            depth += 1
        elif s[i] == ')':
            depth -= 1
        i += 1
    inner = s[start:i-1]
    sm = re.search(r'split\s*=\s*data_args\.(\w+_split_name)', inner)
    if not sm:
        continue
    new_parts.append(s[last_end:m.start()])
    new_parts.append(
        f'{lhs} = (load_from_disk(data_args.dataset_name)[data_args.{sm.group(1)}] '
        f'if os.path.isdir(data_args.dataset_name) and os.path.exists(os.path.join(data_args.dataset_name, "dataset_dict.json")) '
        f'else load_dataset({inner}))'
    )
    last_end = i
new_parts.append(s[last_end:])
s = ''.join(new_parts)

if s != orig:
    with open(script_path, 'w') as f:
        f.write(s)
    print("  run_vits_finetuning.py: patches applied")
else:
    print("  run_vits_finetuning.py: already patched")

print("\nAll patches done.")

### 4. Extend tokenizer + resize embeddings

Bribri uses 5 tone-marking Unicode combining marks that don't appear in the
cjp or spa base tokenizer's vocabulary:
- U+0300 (grave accent), U+0301 (acute), U+0302 (circumflex),
- U+0303 (tilde), U+0308 (diaeresis)

After NFD normalization, these appear as separate codepoints from the base
vowel. We add them as new tokens and resize the text-encoder embedding layer
to match. Existing weights are preserved; new tokens start with random embeddings
that get learned during fine-tuning.

In [ ]:
# 4.1 — Load base, extend tokenizer, resize embeddings, save
base_model = VitsModel.from_pretrained(BASE_MODEL)
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
old_vocab = len(base_tokenizer.get_vocab())
old_embed_rows = base_model.text_encoder.embed_tokens.weight.shape[0]
print(f"Loaded {BASE_MODEL}: vocab={old_vocab}, embed_rows={old_embed_rows}")

COMBINING_MARKS = ['\u0300', '\u0301', '\u0302', '\u0303', '\u0308']
new_marks = [t for t in COMBINING_MARKS if t not in base_tokenizer.get_vocab()]
added = base_tokenizer.add_tokens(new_marks)
new_vocab = len(base_tokenizer.get_vocab())
print(f"Added {added} new tokens → vocab={new_vocab}")

# Resize embeddings, preserving existing rows
old_embed = base_model.text_encoder.embed_tokens
new_embed = nn.Embedding(new_vocab, old_embed.weight.shape[1])
with torch.no_grad():
    new_embed.weight[:old_embed_rows] = old_embed.weight
base_model.text_encoder.embed_tokens = new_embed
base_model.config.vocab_size = new_vocab

os.makedirs(BASE_CKPT_DIR, exist_ok=True)
base_model.save_pretrained(BASE_CKPT_DIR)
base_tokenizer.save_pretrained(BASE_CKPT_DIR)
print(f"Saved extended base → {BASE_CKPT_DIR}")

### 5. Discriminator setup + merge

VITS adversarial training needs both a generator and a discriminator. The
MMS-TTS HuggingFace checkpoint only contains the generator. We convert the
discriminator from the original VITS authors' release using the script in
finetune-hf-vits, then merge gen + disc into a single checkpoint by taking
our extended generator's `embed_tokens` and the temp directory's discriminator
weights and everything else.

In [ ]:
# 5.1 — Convert discriminator + merge with extended generator
if os.path.isdir(TEMP_DISC_DIR):
    shutil.rmtree(TEMP_DISC_DIR)

r = subprocess.run(
    ['python', f'{REPO_DIR}/convert_original_discriminator_checkpoint.py',
     '--language_code', MODEL_TAG,
     '--pytorch_dump_folder_path', TEMP_DISC_DIR],
    capture_output=True, text=True,
)
if r.returncode != 0:
    print(r.stderr[-2000:])
    raise RuntimeError("Discriminator conversion failed")
print(f"Converted disc → {TEMP_DISC_DIR}")

# Merge: temp_state has the right gen+disc structure but wrong-sized embed_tokens.
# Override embed_tokens with our extended one; everything else stays from temp.
temp_state = load_file(f'{TEMP_DISC_DIR}/model.safetensors')
ext_state  = load_file(f'{BASE_CKPT_DIR}/model.safetensors')
for k in ext_state:
    if k in temp_state and (ext_state[k].shape == temp_state[k].shape or 'embed_tokens' in k):
        temp_state[k] = ext_state[k]

os.makedirs(TRAIN_BASE_DIR, exist_ok=True)
# CRITICAL: metadata={"format": "pt"} is required by transformers.from_pretrained
save_file(temp_state, f'{TRAIN_BASE_DIR}/model.safetensors', metadata={"format": "pt"})
save_file(ext_state, f'{BASE_CKPT_DIR}/model.safetensors', metadata={"format": "pt"})

# Config from temp (has discriminator fields); bump vocab_size to our extended value
shutil.copy(f'{TEMP_DISC_DIR}/config.json', f'{TRAIN_BASE_DIR}/config.json')
with open(f'{TRAIN_BASE_DIR}/config.json') as f:
    cfg = json.load(f)
cfg['vocab_size'] = new_vocab
with open(f'{TRAIN_BASE_DIR}/config.json', 'w') as f:
    json.dump(cfg, f, indent=2)

# Tokenizer files come from our extended base; preprocessor from temp
for fn in ['tokenizer_config.json', 'vocab.json', 'added_tokens.json', 'special_tokens_map.json']:
    src = f'{BASE_CKPT_DIR}/{fn}'
    if os.path.exists(src):
        shutil.copy(src, f'{TRAIN_BASE_DIR}/{fn}')
if os.path.exists(f'{TEMP_DISC_DIR}/preprocessor_config.json'):
    shutil.copy(f'{TEMP_DISC_DIR}/preprocessor_config.json', f'{TRAIN_BASE_DIR}/preprocessor_config.json')

print(f"Merged base + disc → {TRAIN_BASE_DIR}")

### 6. Build HF dataset

Convert our pandas splits into a HuggingFace `DatasetDict` with `audio` + `text`
columns and save to Drive. Audio paths only — files are loaded lazily at train
time. The save step takes ~2 min because Drive's `os.path.getsize` is slow
per-file and `datasets.save_to_disk` calls it on each clip to estimate the
shard size.

In [ ]:
# 6.1 — Build and save dataset (~2 min due to Drive getsize per-file)
def df_to_ds(df):
    return Dataset.from_dict({
        'audio': df['path'].tolist(),
        'text':  df['sentence_nfd'].tolist(),
    }).cast_column('audio', Audio(sampling_rate=16000))

ds = DatasetDict({
    'train':      df_to_ds(splits['train']),
    'validation': df_to_ds(splits['valid']),
    'test':       df_to_ds(splits['test']),
})
ds.save_to_disk(DATASET_DIR)
print(f"Saved → {DATASET_DIR}")
print(ds)

### 7. Training configuration

In the config file below, you can tune the training format and parameters. In the end, we found the best performance when reducing the number of epochs (from 100 down to 30), and prioritizing `weight_mel` (trains towards matching the waveform) and reducing `weight_disc` (which trains towards tricking the discriminator, producing "more-human" output, which sounded closer to the audio files we were provided, but not actually intelligible phenomes)

In [ ]:
# 7.1 — Write training config JSON
sample_text = unicodedata.normalize('NFD', "ìs be' shkẽ̀nã")

config = {
    "project_name": f"bribri_vits_{MODEL_TAG}",
    "push_to_hub": False,
    "overwrite_output_dir": True,
    "output_dir": TRAIN_OUTPUT_DIR,

    "dataset_name": DATASET_DIR,
    "dataset_config_name": None,
    "audio_column_name": "audio",
    "text_column_name": "text",
    "train_split_name": "train",
    "eval_split_name": "validation",
    "speaker_id_column_name": None,
    "override_speaker_embeddings": False,

    "full_generation_sample_text": sample_text,

    "max_duration_in_seconds": 20,
    "min_duration_in_seconds": 1.0,
    "max_tokens_length": 500,

    "model_name_or_path": TRAIN_BASE_DIR,
    "preprocessing_num_workers": 2,

    "do_train": True,
    "num_train_epochs": 30,
    "gradient_accumulation_steps": 1,
    "gradient_checkpointing": False,
    "per_device_train_batch_size": 8,
    "learning_rate": 1e-5,
    "adam_beta1": 0.8,
    "adam_beta2": 0.99,
    "warmup_ratio": 0.01,
    "group_by_length": False,

    "do_eval": True,
    "eval_steps": 150,
    "per_device_eval_batch_size": 8,
    "max_eval_samples": 16,
    "do_step_schedule_per_epoch": True,

    "save_steps": 200,
    "save_total_limit": 2,

    "weight_disc": .25,
    "weight_fmaps": 1,
    "weight_gen": 1,
    "weight_kl": 1.5,
    "weight_duration": 1,
    "weight_mel": 65,

    "fp16": True,
    "seed": SEED,
}
os.makedirs(TRAIN_OUTPUT_DIR, exist_ok=True)
with open(CONFIG_PATH, 'w') as f:
    json.dump(config, f, indent=2)
print(f"Wrote {CONFIG_PATH}")

## 8. Launch training

We first run a patch to the imports and loading of the dataset. Checkpoints happen every 200 steps and land in `outputs/{tag}/training_run/checkpoint-N`.

Expect ~25–35 min for 30 epochs on a free-tier T4.

In [ ]:
script_path = f'{REPO_DIR}/run_vits_finetuning.py'
with open(script_path) as f: s = f.read()
if 'import load_dataset, load_from_disk' not in s and 'load_from_disk,' not in s:
    s = s.replace('from datasets import DatasetDict, load_dataset',
                  'from datasets import DatasetDict, load_dataset, load_from_disk')
    with open(script_path, 'w') as f: f.write(s)
    print("Fixed")
else:
    print("Already ok")
!grep -n "from datasets" {REPO_DIR}/run_vits_finetuning.py

In [ ]:
# 8.1 — Launch
!cd {REPO_DIR} && WANDB_DISABLED=true python run_vits_finetuning.py {CONFIG_PATH}

## 9. Inference

Load the fine-tuned model from the training output directory and synthesize the
test sample sentence. The `Some weights of VitsModel were not initialized`
warning about discriminator weights is expected — inference uses only the
generator.

In [ ]:
# 9.1 — Load fine-tuned model and synthesize the test sample
ft_model = VitsModel.from_pretrained(TRAIN_OUTPUT_DIR).to('cuda')
ft_tokenizer = AutoTokenizer.from_pretrained(TRAIN_OUTPUT_DIR)
ft_model.eval()

text = "ìs be' shkẽ̀nã"   # "how are you?" in Bribri
text_nfd = unicodedata.normalize('NFD', text)
inputs = ft_tokenizer(text_nfd, return_tensors="pt").to('cuda')

with torch.no_grad():
    out = ft_model(**inputs)

waveform = out.waveform.squeeze().cpu().numpy()
sr = ft_model.config.sampling_rate
print(f"{waveform.shape[0]} samples @ {sr} Hz = {waveform.shape[0]/sr:.2f}s")
print(f"Fine-tuned ({MODEL_TAG}):")
ipd.display(ipd.Audio(waveform, rate=sr))

# Save for comparison
os.makedirs(SAMPLES_DIR, exist_ok=True)
scipy.io.wavfile.write(f'{SAMPLES_DIR}/_demo_finetuned.wav', rate=sr, data=waveform)

In [ ]:
# 9.2 — Compare with zero-shot baseline
base_zs = VitsModel.from_pretrained(BASE_MODEL).to('cuda')
base_tok = AutoTokenizer.from_pretrained(BASE_MODEL)
base_zs.eval()

inp_b = base_tok(text_nfd, return_tensors="pt").to('cuda')
with torch.no_grad():
    out_b = base_zs(**inp_b)
wav_b = out_b.waveform.squeeze().cpu().numpy()

print(f"Zero-shot baseline ({BASE_MODEL}):")
ipd.display(ipd.Audio(wav_b, rate=base_zs.config.sampling_rate))
scipy.io.wavfile.write(f'{SAMPLES_DIR}/_demo_baseline.wav', rate=base_zs.config.sampling_rate, data=wav_b)

## 10. Generate 20 fixed samples for human evaluation

Per the methodology and our prof's request: 20 samples per algorithm for the
human-eval (MOS) survey. We use a fixed seed and sort the test set by length to
get the same 20 sentences across both the cjp and spa runs (and later
Spark-TTS), so listeners can do direct A/B comparisons per sentence.

We generate from both the fine-tuned model and the zero-shot baseline for the
same 20 sentences, save to `samples_for_eval/`.

In [ ]:
# 10.1 — Pick 20 test sentences (deterministic across all runs)
import numpy as np

test_df = splits['test'].copy()
test_df['len'] = test_df['sentence_nfd'].str.len()
# Sort by length, then take a stride to span short/medium/long sentences
sorted_df = test_df.sort_values('len').reset_index(drop=True)
stride = max(1, len(sorted_df) // 20)
chosen = sorted_df.iloc[::stride].head(20).reset_index(drop=True)
print(f"Selected {len(chosen)} sentences (lengths {chosen['len'].min()}–{chosen['len'].max()})")
print(chosen[['sentence', 'len']].head(5).to_string())

In [ ]:
# 10.2 — Synthesize and save 20 samples from both models
os.makedirs(f'{SAMPLES_DIR}/finetuned', exist_ok=True)
os.makedirs(f'{SAMPLES_DIR}/baseline', exist_ok=True)

# Sentence manifest (CSV) — listeners need to know what was supposed to be said
manifest_rows = []

for i, row in chosen.iterrows():
    text_nfd_i = row['sentence_nfd']
    fname = f"sample_{i:02d}.wav"

    # Fine-tuned
    inp_ft = ft_tokenizer(text_nfd_i, return_tensors="pt").to('cuda')
    with torch.no_grad():
        out_ft = ft_model(**inp_ft)
    wav_ft = out_ft.waveform.squeeze().cpu().numpy()
    scipy.io.wavfile.write(f'{SAMPLES_DIR}/finetuned/{fname}', rate=ft_model.config.sampling_rate, data=wav_ft)

    # Baseline
    inp_b = base_tok(text_nfd_i, return_tensors="pt").to('cuda')
    with torch.no_grad():
        out_b = base_zs(**inp_b)
    wav_b = out_b.waveform.squeeze().cpu().numpy()
    scipy.io.wavfile.write(f'{SAMPLES_DIR}/baseline/{fname}', rate=base_zs.config.sampling_rate, data=wav_b)

    manifest_rows.append({
        'id': i,
        'filename': fname,
        'sentence_raw': row['sentence'],
        'sentence_nfd': text_nfd_i,
        'duration_ft':       wav_ft.shape[0] / ft_model.config.sampling_rate,
        'duration_baseline': wav_b.shape[0]  / base_zs.config.sampling_rate,
    })

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(f'{SAMPLES_DIR}/manifest.csv', index=False)
print(f"\nWrote {len(manifest)} samples + manifest → {SAMPLES_DIR}")
print(manifest[['id', 'filename', 'sentence_raw']].to_string())